In [3]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error
from pyod.models.auto_encoder import AutoEncoder

# import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
!pip3 install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128


In [4]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [5]:
DATA_FOLDER = 'KukaVelocityDataset'

col_names = np.load(os.path.join(DATA_FOLDER, 'KukaColumnNames.npy'), allow_pickle=True)
kuka_normal = np.load(os.path.join(DATA_FOLDER, 'KukaNormal.npy'), allow_pickle=True)
kuka_slow = np.load(os.path.join(DATA_FOLDER, 'KukaSlow.npy'), allow_pickle=True)

In [6]:
df_kuka_normal = pd.DataFrame(kuka_normal, columns=col_names[:-1])
df_kuka_slow = pd.DataFrame(kuka_slow, columns=col_names)

In [7]:
df_kuka_slow

,action,machine_nameKuka Robot_apparent_power,machine_nameKuka Robot_current,machine_nameKuka Robot_frequency,machine_nameKuka Robot_phase_angle,machine_nameKuka Robot_power,machine_nameKuka Robot_power_factor,machine_nameKuka Robot_reactive_power,machine_nameKuka Robot_voltage,sensor_id1_AccX,...,sensor_id7_AccZ,sensor_id7_GyroX,sensor_id7_GyroY,sensor_id7_GyroZ,sensor_id7_q1,sensor_id7_q2,sensor_id7_q3,sensor_id7_q4,sensor_id7_temp,anomaly
0,10.0,216.610596,1.201105,50.000000,330.759125,189.012650,0.872592,-105.793724,228.445389,-0.629883,...,1.022461,-0.976562,-9.033203,40.222168,-0.210682,0.013827,-0.336950,0.917544,180.24,1.0
1,10.0,216.610596,1.201105,50.000000,330.759125,189.012650,0.872592,-105.793724,228.445389,-0.629883,...,1.012695,-0.183105,-2.075195,6.713867,-0.211322,0.010658,-0.317552,0.924332,180.24,1.0
2,10.0,216.610596,1.201105,49.951218,330.759125,189.012650,0.872592,-105.793724,228.445389,-0.490234,...,1.019531,0.000000,-0.183105,-0.122070,-0.208518,0.014488,-0.316478,0.925285,180.24,1.0
3,10.0,216.610596,1.201105,49.951218,330.759125,189.012650,0.872592,-105.793724,228.445389,-0.623535,...,0.989258,0.061035,0.183105,-0.305176,-0.206040,0.017369,-0.316144,0.925904,180.24,1.0
4,10.0,216.610596,1.201105,49.951218,330.759125,189.012650,0.872592,-105.793724,228.445389,-0.591797,...,0.987793,-0.122070,0.122070,0.671387,-0.203967,0.019725,-0.315832,0.926423,180.24,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41533,0.0,216.718460,1.265179,49.951218,329.322937,186.394455,0.860077,-110.552055,228.759598,-0.598633,...,1.011719,0.000000,-9.826660,42.175293,-0.156446,-0.108547,0.391621,0.900209,180.24,1.0
41534,0.0,216.718460,1.265179,49.951218,329.322937,186.394455,0.860077,-110.552055,228.759598,-0.598633,...,0.991211,-0.122070,-9.704590,41.137695,-0.152054,-0.120020,0.422892,0.885233,180.24,1.0
41535,0.0,216.718460,1.265179,49.951218,329.322937,186.394455,0.860077,-110.552055,228.759598,-0.598633,...,1.013672,0.000000,-9.643555,41.992188,-0.147580,-0.130558,0.452075,0.869944,180.24,1.0
41536,0.0,216.718460,1.265179,49.951218,329.322937,186.394455,0.860077,-110.552055,228.759598,-0.598633,...,0.997559,-0.488281,-9.948730,40.527344,-0.142647,-0.141307,0.482355,0.852654,180.24,1.0


In [8]:
# removing the anomaly feature from the slow dataset
if "anomaly" in df_kuka_slow.columns:
    df_slow_features = df_kuka_slow.drop(columns=["anomaly"])
else:
    df_slow_features = df_kuka_slow.copy()

df_normal_features = df_kuka_normal.copy()

In [9]:
# Standardize the features
# scaler = StandardScaler()
scaler = MinMaxScaler(feature_range=(0, 1))
# fit on globar standard
scaler.fit(df_normal_features)

X_normal_scaled = scaler.transform(df_normal_features)
X_slow_scaled   = scaler.transform(df_slow_features)

In [10]:
# creating sliding windows of size 50
WINDOW_SIZE = 50   # 5 seconds at 10 Hz
STRIDE = 1

def create_windows(data, window_size, stride=1):
    windows = []
    for i in range(0, len(data) - window_size + 1, stride):
        windows.append(data[i:i + window_size])
    return np.array(windows)

In [11]:
X_train = create_windows(X_normal_scaled, WINDOW_SIZE, STRIDE)
X_test  = create_windows(X_slow_scaled, WINDOW_SIZE, STRIDE)

print("Train windows:", X_train.shape)
print("Test windows: ", X_test.shape)


Train windows: (233743, 50, 86)
Test windows:  (41489, 50, 86)


In [12]:
# flattening windows for pyOD
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

print("Flattened shape:", X_train_flat.shape)

Flattened shape: (233743, 4300)


In [13]:
import torch
from pyod.models.auto_encoder import AutoEncoder
from joblib import dump, load

# Check if CUDA is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Initialize the AutoEncoder
ae = AutoEncoder(
    hidden_neuron_list=[256, 64, 256],
    hidden_activation_name='relu',
    epoch_num=50,
    batch_size=64,
    lr=1e-3,
    contamination=0.001,
    batch_norm=True,
    dropout_rate=0.2,
    verbose=1,
    # Use GPU if available
    device=device,
    compile_mode='auto'
)

# Fit the model
# ae.fit(X_train)

# --- MODEL PERSISTENCE ---

# 1. Save the entire model object using joblib
model_filename = 'autoencoder_model.joblib'
dump(ae, model_filename)
print(f"Model saved to {model_filename}")

# 2. To Load the model later:
# loaded_ae = load(model_filename)

Using device: cuda
Model saved to autoencoder_model.joblib


In [13]:
ae.device

'cuda'

In [14]:
ae.fit(X_train_flat)

Training: 100%|██████████| 50/50 [1:57:08<00:00, 140.58s/it]   


In [15]:
train_scores = ae.decision_scores_                 # Normal windows
test_scores  = ae.decision_function(X_test_flat)  # Slow windows

In [14]:
# ae.save('sample_autoencoder_model.pth')
# 2. To Load the model later:
file_name = 'sample_autoencoder_model.pth'
loaded_ae = load(model_filename)

In [15]:
# predicting on test data
y_test_pred = ae.predict(X_test_flat)  # 0 for normal, 1 for anomalous
print("Predicted labels for test data:", y_test_pred)
y_test_pred

NotFittedError: This AutoEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [22]:
sum(y_test_pred)  # number of predicted anomalies in the test set

196

In [2]:
y_true = np.ones_like(y_test_pred)

NameError: name 'np' is not defined

In [1]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_true, y_test_pred))
print(classification_report(
    y_true,
    y_test_pred,
    target_names=["Normal", "Anomaly"]
))

NameError: name 'y_true' is not defined